In [1]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw")
CLEAN_DIR = Path("../data/cleaned")

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print("Golden dataset workspace ready.")

Golden dataset workspace ready.


In [2]:
# STEP 11A: Create cleaned payments

payments = pd.read_csv(RAW_DIR / "payments.csv")

raw_rows = len(payments)

clean_payments = payments.drop_duplicates(
    subset=["payment_id"],
    keep="first"
).copy()

clean_rows = len(clean_payments)

print("Raw payment rows:", raw_rows)
print("Clean payment rows:", clean_rows)
print("Rows removed:", raw_rows - clean_rows)

raw_success = payments.loc[
    payments["payment_status"] == "SUCCESS", "amount"
].sum()

clean_success = clean_payments.loc[
    clean_payments["payment_status"] == "SUCCESS", "amount"
].sum()

print(f"Raw SUCCESS recovery: ₹{raw_success:,.2f}")
print(f"Clean SUCCESS recovery: ₹{clean_success:,.2f}")
print(f"Recovery difference: ₹{raw_success-clean_success:,.2f}")

Raw payment rows: 25500
Clean payment rows: 25000
Rows removed: 500
Raw SUCCESS recovery: ₹1,341,485,926.33
Clean SUCCESS recovery: ₹1,315,583,964.64
Recovery difference: ₹25,901,961.69


In [3]:
# STEP 11B: Create cleaned accounts table

accounts = pd.read_csv(RAW_DIR / "accounts.csv")
borrowers = pd.read_csv(RAW_DIR / "borrowers.csv")

valid_borrower_ids = set(
    borrowers["borrower_id"].dropna().unique()
)

# Classify the borrower relationship
accounts["borrower_link_status"] = "VALID"

accounts.loc[
    accounts["borrower_id"].isna(),
    "borrower_link_status"
] = "MISSING_BORROWER_ID"

accounts.loc[
    accounts["borrower_id"].notna()
    & ~accounts["borrower_id"].isin(valid_borrower_ids),
    "borrower_link_status"
] = "INVALID_BORROWER_ID"

print("Account rows:", len(accounts))
print("\nBorrower link status:")
display(
    accounts["borrower_link_status"]
    .value_counts()
    .to_frame("count")
)

print("\nPercent of accounts:")
display(
    (
        accounts["borrower_link_status"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .to_frame("percentage")
    )
)

Account rows: 30000

Borrower link status:


,count
borrower_link_status,
VALID,27087
INVALID_BORROWER_ID,2458
MISSING_BORROWER_ID,455



Percent of accounts:


,percentage
borrower_link_status,
VALID,90.29
INVALID_BORROWER_ID,8.19
MISSING_BORROWER_ID,1.52


In [4]:
accounts.to_csv(
    CLEAN_DIR / "accounts_clean.csv",
    index=False
)

print("Saved:", CLEAN_DIR / "accounts_clean.csv")

Saved: ..\data\cleaned\accounts_clean.csv


In [5]:
# STEP 11C: Clean calls and normalize timestamps

calls = pd.read_csv(RAW_DIR / "calls.csv")

raw_call_rows = len(calls)
exact_duplicate_calls = calls.duplicated().sum()

# Remove only exact duplicate rows
clean_calls = calls.drop_duplicates().copy()

# Preserve original timestamp
clean_calls["event_at_local"] = pd.to_datetime(
    clean_calls["event_at"],
    errors="coerce"
)

# Convert each row using its own timezone
clean_calls["event_at_utc"] = [
    ts.tz_localize(tz).tz_convert("UTC")
    for ts, tz in zip(
        clean_calls["event_at_local"],
        clean_calls["timezone"]
    )
]

# Common analytical timezone
clean_calls["event_at_ist"] = (
    clean_calls["event_at_utc"]
    .dt.tz_convert("Asia/Kolkata")
)

print("Raw call rows:", raw_call_rows)
print("Exact duplicate rows:", exact_duplicate_calls)
print("Clean call rows:", len(clean_calls))
print("Rows removed:", raw_call_rows - len(clean_calls))

print("\nMissing normalized timestamps:")
print(clean_calls["event_at_ist"].isna().sum())

Raw call rows: 91350
Exact duplicate rows: 1271
Clean call rows: 90079
Rows removed: 1271

Missing normalized timestamps:
0


In [6]:
clean_calls.to_csv(
    CLEAN_DIR / "calls_clean.csv",
    index=False
)

print("Saved:", CLEAN_DIR / "calls_clean.csv")

Saved: ..\data\cleaned\calls_clean.csv


In [7]:
# STEP 11D: Investigate borrower duplicates

borrowers = pd.read_csv(RAW_DIR / "borrowers.csv")

print("Raw borrower rows:", len(borrowers))
print("Exact duplicate rows:", borrowers.duplicated().sum())
print("Unique borrower IDs:", borrowers["borrower_id"].nunique())

duplicate_borrowers = borrowers[
    borrowers.duplicated(keep=False)
].sort_values("borrower_id")

print("\nRows involved in duplicate records:",
      len(duplicate_borrowers))

display(duplicate_borrowers.head(20))

Raw borrower rows: 30600
Exact duplicate rows: 600
Unique borrower IDs: 11015

Rows involved in duplicate records: 1200


,borrower_id,name,phone,email,city,created_at,updated_at,state
30325,BRW0000005,Neha Singh,9.269813e+09,user4520@example.com,Mumbai,2026-02-07 14:08:27,2025-12-04 08:52:09,Maharashtra
18692,BRW0000005,Neha Singh,9.269813e+09,user4520@example.com,Mumbai,2026-02-07 14:08:27,2025-12-04 08:52:09,Maharashtra
30523,BRW0000064,Neha Singh,9.974780e+09,user13812@example.com,Delhi,2026-04-23 11:10:41,2025-04-20 23:34:36,Delhi
19635,BRW0000064,Neha Singh,9.974780e+09,user13812@example.com,Delhi,2026-04-23 11:10:41,2025-04-20 23:34:36,Delhi
30328,BRW0000078,Ananya Rao,9.400819e+09,user2170@example.com,Bengaluru,2025-03-09 20:53:26,2026-07-06 03:48:59,Karnataka
21400,BRW0000078,Ananya Rao,9.400819e+09,user2170@example.com,Bengaluru,2025-03-09 20:53:26,2026-07-06 03:48:59,Karnataka
30454,BRW0000113,Aarav Sharma,9.246415e+09,user7854@example.com,Bhubaneswar,2026-03-12 02:20:13,2026-04-13 12:38:53,Odisha
11843,BRW0000113,Aarav Sharma,9.246415e+09,user7854@example.com,Bhubaneswar,2026-03-12 02:20:13,2026-04-13 12:38:53,Odisha
12630,BRW0000122,Amit Kumar,9.119349e+09,user14099@example.com,Hyderabad,2025-10-12 15:11:11,2025-09-07 02:25:29,Telangana
30163,BRW0000122,Amit Kumar,9.119349e+09,user14099@example.com,Hyderabad,2025-10-12 15:11:11,2025-09-07 02:25:29,Telangana


In [8]:
# STEP 11D: Create cleaned borrowers table

clean_borrowers = borrowers.drop_duplicates().copy()

print("Raw borrower rows:", len(borrowers))
print("Clean borrower rows:", len(clean_borrowers))
print("Rows removed:", len(borrowers) - len(clean_borrowers))
print("Unique borrower IDs after cleaning:",
      clean_borrowers["borrower_id"].nunique())

Raw borrower rows: 30600
Clean borrower rows: 30000
Rows removed: 600
Unique borrower IDs after cleaning: 11015


In [9]:
clean_borrowers.to_csv(
    CLEAN_DIR / "borrowers_clean.csv",
    index=False
)

print("Saved:", CLEAN_DIR / "borrowers_clean.csv")

Saved: ..\data\cleaned\borrowers_clean.csv


In [10]:
# STEP 11E: Inspect daily targeting

targeting = pd.read_csv(
    RAW_DIR / "daily_targeting.csv"
)

print("Raw targeting rows:", len(targeting))
print("Columns:")
print(targeting.columns.tolist())

print("\nExact duplicate rows:",
      targeting.duplicated().sum())

print("\nSample:")
display(targeting.head(10))

Raw targeting rows: 45000
Columns:
['target_id', 'account_id', 'campaign_id', 'target_date', 'priority', 'recommended_channel', 'status']

Exact duplicate rows: 0

Sample:


,target_id,account_id,campaign_id,target_date,priority,recommended_channel,status
0,TGT0000001,ACC0028555,CMP0000103,2026-04-22,4,FIELD,QUEUED
1,TGT0000002,ACC0007194,CMP0000100,2026-08-06,10,SMS,EXPIRED
2,TGT0000003,ACC0029550,CMP0000074,2026-05-20,5,SMS,CONTACTED
3,TGT0000004,ACC0012329,CMP0000046,2026-07-30,10,WHATSAPP,QUEUED
4,TGT0000005,ACC0018387,CMP0000118,2026-07-10,7,WHATSAPP,EXPIRED
5,TGT0000006,ACC0002016,CMP0000024,2026-03-07,6,FIELD,QUEUED
6,TGT0000007,ACC0019181,CMP0000038,2026-04-20,7,SMS,QUEUED
7,TGT0000008,ACC0004874,CMP0000107,2026-02-19,1,FIELD,EXPIRED
8,TGT0000009,ACC0012866,CMP0000109,2026-06-07,8,SMS,CONTACTED
9,TGT0000010,ACC0027621,CMP0000092,2026-06-07,8,WHATSAPP,SKIPPED


In [11]:
# STEP 11E: Validate daily targeting

targeting = pd.read_csv(
    RAW_DIR / "daily_targeting.csv"
)

print("Total targeting rows:", len(targeting))
print("Unique target_id:", targeting["target_id"].nunique())
print(
    "Duplicate target_id rows:",
    targeting["target_id"].duplicated().sum()
)

print("\nMissing values:")
print(targeting.isna().sum())

print("\nTargeting date range:")
targeting["target_date"] = pd.to_datetime(
    targeting["target_date"],
    errors="coerce"
)

print("Minimum:", targeting["target_date"].min())
print("Maximum:", targeting["target_date"].max())

print("\nRecommended channel distribution:")
display(
    targeting["recommended_channel"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nTargeting status distribution:")
display(
    targeting["status"]
    .value_counts(dropna=False)
    .to_frame("count")
)

Total targeting rows: 45000
Unique target_id: 45000
Duplicate target_id rows: 0

Missing values:
target_id              0
account_id             0
campaign_id            0
target_date            0
priority               0
recommended_channel    0
status                 0
dtype: int64

Targeting date range:
Minimum: 2026-01-01 00:00:00
Maximum: 2026-08-08 00:00:00

Recommended channel distribution:


,count
recommended_channel,
FIELD,11365
SMS,11221
WHATSAPP,11212
VOICE,11202



Targeting status distribution:


,count
status,
EXPIRED,11371
CONTACTED,11254
QUEUED,11202
SKIPPED,11173


In [12]:
# STEP 11E: Save cleaned daily targeting

clean_targeting = targeting.copy()

clean_targeting.to_csv(
    CLEAN_DIR / "daily_targeting_clean.csv",
    index=False
)

print("Saved:", CLEAN_DIR / "daily_targeting_clean.csv")

Saved: ..\data\cleaned\daily_targeting_clean.csv


In [13]:
# STEP 11F: Inspect promises_to_pay

ptp = pd.read_csv(
    RAW_DIR / "promises_to_pay.csv"
)

print("Raw PTP rows:", len(ptp))

print("\nColumns:")
print(ptp.columns.tolist())

print("\nExact duplicate rows:",
      ptp.duplicated().sum())

print("\nMissing values:")
print(ptp.isna().sum())

print("\nSample records:")
display(ptp.head(10))

Raw PTP rows: 18000

Columns:
['ptp_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'promised_amount', 'promised_date', 'status', 'source']

Exact duplicate rows: 0

Missing values:
ptp_id             0
account_id         0
borrower_id        0
event_at           0
agent_id           0
promised_amount    0
promised_date      0
status             0
source             0
dtype: int64

Sample records:


,ptp_id,account_id,borrower_id,event_at,agent_id,promised_amount,promised_date,status,source
0,PTP0000001,ACC0021078,BRW0003789,2026-04-18 21:11:02,AGT0000787,93575.19,2026-05-17 21:11:02,CANCELLED,FIELD
1,PTP0000002,ACC0026620,BRW0010192,2026-07-08 21:49:28,AGT0000763,83558.92,2026-07-25 21:49:28,KEPT,WHATSAPP
2,PTP0000003,ACC0018900,BRW0011197,2026-08-08 08:57:34,AGT0000878,18164.89,2026-09-02 08:57:34,OPEN,SMS
3,PTP0000004,ACC0019107,BRW0009681,2026-01-11 06:04:20,AGT0000535,52829.08,2026-01-24 06:04:20,KEPT,WHATSAPP
4,PTP0000005,ACC0014157,BRW0004907,2026-05-08 01:42:07,AGT0000115,74970.57,2026-05-19 01:42:07,BROKEN,SMS
5,PTP0000006,ACC0020223,BRW0004429,2026-06-14 00:43:18,AGT0000243,51103.49,2026-06-19 00:43:18,BROKEN,WHATSAPP
6,PTP0000007,ACC0018006,BRW0008673,2026-06-04 12:43:41,AGT0000973,18575.15,2026-06-24 12:43:41,CANCELLED,CALL
7,PTP0000008,ACC0011737,BRW0007288,2026-05-17 14:25:29,AGT0000044,64326.80,2026-05-31 14:25:29,OPEN,CALL
8,PTP0000009,ACC0021229,BRW0007682,2026-08-02 10:41:38,AGT0000149,56608.95,2026-08-15 10:41:38,BROKEN,WHATSAPP
9,PTP0000010,ACC0028901,BRW0009589,2026-02-05 15:34:26,AGT0000110,55403.24,2026-02-20 15:34:26,KEPT,SMS


In [14]:
# STEP 11F: Inspect promises_to_pay

ptp = pd.read_csv(
    RAW_DIR / "promises_to_pay.csv"
)

print("Raw PTP rows:", len(ptp))

print("\nColumns:")
print(ptp.columns.tolist())

print("\nExact duplicate rows:",
      ptp.duplicated().sum())

print("\nMissing values:")
print(ptp.isna().sum())

print("\nSample records:")
display(ptp.head(10))

Raw PTP rows: 18000

Columns:
['ptp_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'promised_amount', 'promised_date', 'status', 'source']

Exact duplicate rows: 0

Missing values:
ptp_id             0
account_id         0
borrower_id        0
event_at           0
agent_id           0
promised_amount    0
promised_date      0
status             0
source             0
dtype: int64

Sample records:


,ptp_id,account_id,borrower_id,event_at,agent_id,promised_amount,promised_date,status,source
0,PTP0000001,ACC0021078,BRW0003789,2026-04-18 21:11:02,AGT0000787,93575.19,2026-05-17 21:11:02,CANCELLED,FIELD
1,PTP0000002,ACC0026620,BRW0010192,2026-07-08 21:49:28,AGT0000763,83558.92,2026-07-25 21:49:28,KEPT,WHATSAPP
2,PTP0000003,ACC0018900,BRW0011197,2026-08-08 08:57:34,AGT0000878,18164.89,2026-09-02 08:57:34,OPEN,SMS
3,PTP0000004,ACC0019107,BRW0009681,2026-01-11 06:04:20,AGT0000535,52829.08,2026-01-24 06:04:20,KEPT,WHATSAPP
4,PTP0000005,ACC0014157,BRW0004907,2026-05-08 01:42:07,AGT0000115,74970.57,2026-05-19 01:42:07,BROKEN,SMS
5,PTP0000006,ACC0020223,BRW0004429,2026-06-14 00:43:18,AGT0000243,51103.49,2026-06-19 00:43:18,BROKEN,WHATSAPP
6,PTP0000007,ACC0018006,BRW0008673,2026-06-04 12:43:41,AGT0000973,18575.15,2026-06-24 12:43:41,CANCELLED,CALL
7,PTP0000008,ACC0011737,BRW0007288,2026-05-17 14:25:29,AGT0000044,64326.80,2026-05-31 14:25:29,OPEN,CALL
8,PTP0000009,ACC0021229,BRW0007682,2026-08-02 10:41:38,AGT0000149,56608.95,2026-08-15 10:41:38,BROKEN,WHATSAPP
9,PTP0000010,ACC0028901,BRW0009589,2026-02-05 15:34:26,AGT0000110,55403.24,2026-02-20 15:34:26,KEPT,SMS


In [15]:
# STEP 11F: Validate PTP IDs and statuses

print("Total PTP rows:", len(ptp))
print("Unique PTP IDs:", ptp["ptp_id"].nunique())
print(
    "Duplicate PTP IDs:",
    ptp["ptp_id"].duplicated().sum()
)

print("\nPTP status distribution:")
display(
    ptp["status"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nPTP source distribution:")
display(
    ptp["source"]
    .value_counts(dropna=False)
    .to_frame("count")
)

Total PTP rows: 18000
Unique PTP IDs: 18000
Duplicate PTP IDs: 0

PTP status distribution:


,count
status,
BROKEN,4553
CANCELLED,4543
KEPT,4489
OPEN,4415



PTP source distribution:


,count
source,
CALL,4571
FIELD,4546
SMS,4449
WHATSAPP,4434


In [16]:
# STEP 11F: Save cleaned PTP table

clean_ptp = ptp.copy()

clean_ptp.to_csv(
    CLEAN_DIR / "promises_to_pay_clean.csv",
    index=False
)

print("Saved:", CLEAN_DIR / "promises_to_pay_clean.csv")

Saved: ..\data\cleaned\promises_to_pay_clean.csv


In [17]:
# STEP 11G: Inspect field visits

field_visits = pd.read_csv(
    RAW_DIR / "field_visits.csv"
)

print("Raw field-visit rows:", len(field_visits))

print("\nColumns:")
print(field_visits.columns.tolist())

print("\nExact duplicate rows:",
      field_visits.duplicated().sum())

print("\nMissing values:")
print(field_visits.isna().sum())

print("\nSample records:")
display(field_visits.head(10))

Raw field-visit rows: 25000

Columns:
['visit_id', 'account_id', 'borrower_id', 'event_at', 'agent_id', 'visit_type', 'outcome', 'latitude', 'longitude', 'scheduled_at']

Exact duplicate rows: 0

Missing values:
visit_id          0
account_id        0
borrower_id       0
event_at          0
agent_id          0
visit_type        0
outcome           0
latitude          0
longitude         0
scheduled_at    250
dtype: int64

Sample records:


,visit_id,account_id,borrower_id,event_at,agent_id,visit_type,outcome,latitude,longitude,scheduled_at
0,VISIT0000001,ACC0026237,BRW0009469,2026-01-25 01:15:48,AGT0000403,AFTER_HOURS,PAID,31.861706,76.098184,2026-01-24 05:49:28
1,VISIT0000002,ACC0026942,BRW0010461,2026-06-23 22:18:25,AGT0000278,NOTICE,PAID,23.641189,74.571935,2026-06-23 14:17:54
2,VISIT0000003,ACC0022809,BRW0006318,2026-05-16 08:06:34,AGT0000723,NOTICE,CONTACTED,22.324370,84.790537,2026-05-16 02:58:39
3,VISIT0000004,ACC0029193,BRW0005195,2026-06-14 00:12:17,AGT0000946,FOLLOWUP,CONTACTED,27.230510,75.957354,2026-06-13 13:32:15
4,VISIT0000005,ACC0015605,BRW0009229,2026-08-01 21:31:09,AGT0000917,NOTICE,REFUSED,23.563588,85.547376,2026-08-01 00:47:29
5,VISIT0000006,ACC0015482,BRW0001304,2026-05-29 12:40:27,AGT0000372,FIRST_VISIT,NOT_AVAILABLE,29.208521,80.847281,2026-05-29 02:11:57
6,VISIT0000007,ACC0026113,BRW0000922,2026-07-16 22:36:38,AGT0000326,AFTER_HOURS,WRONG_ADDRESS,30.980902,75.520268,2026-07-16 08:41:48
7,VISIT0000008,ACC0013041,BRW0004896,2026-02-21 13:05:03,AGT0000008,FIRST_VISIT,CONTACTED,11.981796,81.715324,2026-02-20 22:14:29
8,VISIT0000009,ACC0002139,BRW0008015,2026-06-13 17:33:43,AGT0000243,AFTER_HOURS,REFUSED,18.746754,88.817738,2026-06-13 17:15:29
9,VISIT0000010,ACC0007229,BRW0008110,2026-07-07 09:34:50,AGT0000959,FOLLOWUP,REFUSED,27.441805,74.835577,2026-07-06 10:55:37


In [18]:
# STEP 11G: Validate field visits

field_visits = pd.read_csv(
    RAW_DIR / "field_visits.csv"
)

print("Total field visits:", len(field_visits))

print("Unique visit IDs:",
      field_visits["visit_id"].nunique())

print(
    "Duplicate visit IDs:",
    field_visits["visit_id"].duplicated().sum()
)

print("\nMissing values by column:")
display(
    field_visits.isna().sum().to_frame("missing_count")
)

print("\nOutcome distribution:")
display(
    field_visits["outcome"]
    .value_counts(dropna=False)
    .to_frame("count")
)


Total field visits: 25000
Unique visit IDs: 25000
Duplicate visit IDs: 0

Missing values by column:


,missing_count
visit_id,0
account_id,0
borrower_id,0
event_at,0
agent_id,0
visit_type,0
outcome,0
latitude,0
longitude,0
scheduled_at,250



Outcome distribution:


,count
outcome,
WRONG_ADDRESS,4221
CONTACTED,4205
PTP,4200
NOT_AVAILABLE,4190
REFUSED,4104
PAID,4080


In [19]:
# STEP 11G: Validate field visits

field_visits = pd.read_csv(
    RAW_DIR / "field_visits.csv"
)

print("Total field visits:", len(field_visits))

print("Unique visit IDs:",
      field_visits["visit_id"].nunique())

print(
    "Duplicate visit IDs:",
    field_visits["visit_id"].duplicated().sum()
)

print("\nMissing values by column:")
display(
    field_visits.isna().sum().to_frame("missing_count")
)

print("\nOutcome distribution:")
display(
    field_visits["outcome"]
    .value_counts(dropna=False)
    .to_frame("count")
)

Total field visits: 25000
Unique visit IDs: 25000
Duplicate visit IDs: 0

Missing values by column:


,missing_count
visit_id,0
account_id,0
borrower_id,0
event_at,0
agent_id,0
visit_type,0
outcome,0
latitude,0
longitude,0
scheduled_at,250



Outcome distribution:


,count
outcome,
WRONG_ADDRESS,4221
CONTACTED,4205
PTP,4200
NOT_AVAILABLE,4190
REFUSED,4104
PAID,4080


In [20]:
# STEP 11G: Save cleaned field visits

clean_field_visits = field_visits.copy()

clean_field_visits.to_csv(
    CLEAN_DIR / "field_visits_clean.csv",
    index=False
)

print("Saved:", CLEAN_DIR / "field_visits_clean.csv")

Saved: ..\data\cleaned\field_visits_clean.csv


In [21]:
# STEP 11H: Inspect SMS events

sms = pd.read_csv(
    RAW_DIR / "sms_events.csv"
)

print("Raw SMS rows:", len(sms))

print("\nColumns:")
print(sms.columns.tolist())

print("\nExact duplicate rows:",
      sms.duplicated().sum())

print("\nMissing values:")
print(sms.isna().sum())

print("\nSample records:")
display(sms.head(10))

Raw SMS rows: 45000

Columns:
['sms_event_id', 'account_id', 'borrower_id', 'event_at', 'message_id', 'event_type', 'template_code', 'provider_id']

Exact duplicate rows: 0

Missing values:
sms_event_id     0
account_id       0
borrower_id      0
event_at         0
message_id       0
event_type       0
template_code    0
provider_id      0
dtype: int64

Sample records:


,sms_event_id,account_id,borrower_id,event_at,message_id,event_type,template_code,provider_id
0,SMS_EVENT0000001,ACC0010751,BRW0009574,2026-04-24 20:57:06,SMS000022764,CLICKED,PAYMENT_LINK,VND0000010
1,SMS_EVENT0000002,ACC0005842,BRW0003619,2026-05-24 11:28:20,SMS000015050,DELIVERED,NOTICE,VND0000003
2,SMS_EVENT0000003,ACC0014253,BRW0003018,2026-03-05 20:23:20,SMS000028337,CLICKED,NOTICE,VND0000008
3,SMS_EVENT0000004,ACC0016374,BRW0002587,2026-05-11 04:56:15,SMS000004875,DELIVERED,DUE_REMINDER,VND0000010
4,SMS_EVENT0000005,ACC0000796,BRW0010499,2026-03-24 08:41:53,SMS000008899,SENT,PTP_REMINDER,VND0000007
5,SMS_EVENT0000006,ACC0009328,BRW0000789,2026-06-11 17:08:08,SMS000035813,CLICKED,PAYMENT_LINK,VND0000010
6,SMS_EVENT0000007,ACC0009624,BRW0011563,2026-07-07 13:52:06,SMS000020047,FAILED,PTP_REMINDER,VND0000011
7,SMS_EVENT0000008,ACC0010884,BRW0008044,2026-06-18 18:40:28,SMS000016839,SENT,DUE_REMINDER,VND0000006
8,SMS_EVENT0000009,ACC0015268,BRW0011110,2026-07-07 18:02:40,SMS000004261,SENT,PAYMENT_LINK,VND0000002
9,SMS_EVENT0000010,ACC0021694,BRW0004461,2026-02-10 08:42:24,SMS000004158,FAILED,PAYMENT_LINK,VND0000014


In [22]:
# STEP 11H: Inspect SMS events

sms = pd.read_csv(
    RAW_DIR / "sms_events.csv"
)

print("Raw SMS rows:", len(sms))

print("\nColumns:")
print(sms.columns.tolist())

print("\nExact duplicate rows:",
      sms.duplicated().sum())

print("\nMissing values:")
print(sms.isna().sum())

print("\nSample records:")
display(sms.head(10))

Raw SMS rows: 45000

Columns:
['sms_event_id', 'account_id', 'borrower_id', 'event_at', 'message_id', 'event_type', 'template_code', 'provider_id']

Exact duplicate rows: 0

Missing values:
sms_event_id     0
account_id       0
borrower_id      0
event_at         0
message_id       0
event_type       0
template_code    0
provider_id      0
dtype: int64

Sample records:


,sms_event_id,account_id,borrower_id,event_at,message_id,event_type,template_code,provider_id
0,SMS_EVENT0000001,ACC0010751,BRW0009574,2026-04-24 20:57:06,SMS000022764,CLICKED,PAYMENT_LINK,VND0000010
1,SMS_EVENT0000002,ACC0005842,BRW0003619,2026-05-24 11:28:20,SMS000015050,DELIVERED,NOTICE,VND0000003
2,SMS_EVENT0000003,ACC0014253,BRW0003018,2026-03-05 20:23:20,SMS000028337,CLICKED,NOTICE,VND0000008
3,SMS_EVENT0000004,ACC0016374,BRW0002587,2026-05-11 04:56:15,SMS000004875,DELIVERED,DUE_REMINDER,VND0000010
4,SMS_EVENT0000005,ACC0000796,BRW0010499,2026-03-24 08:41:53,SMS000008899,SENT,PTP_REMINDER,VND0000007
5,SMS_EVENT0000006,ACC0009328,BRW0000789,2026-06-11 17:08:08,SMS000035813,CLICKED,PAYMENT_LINK,VND0000010
6,SMS_EVENT0000007,ACC0009624,BRW0011563,2026-07-07 13:52:06,SMS000020047,FAILED,PTP_REMINDER,VND0000011
7,SMS_EVENT0000008,ACC0010884,BRW0008044,2026-06-18 18:40:28,SMS000016839,SENT,DUE_REMINDER,VND0000006
8,SMS_EVENT0000009,ACC0015268,BRW0011110,2026-07-07 18:02:40,SMS000004261,SENT,PAYMENT_LINK,VND0000002
9,SMS_EVENT0000010,ACC0021694,BRW0004461,2026-02-10 08:42:24,SMS000004158,FAILED,PAYMENT_LINK,VND0000014


In [23]:
# STEP 11H: Validate SMS event IDs

print("Total SMS rows:", len(sms))

print(
    "Unique sms_event_id:",
    sms["sms_event_id"].nunique()
)

print(
    "Duplicate sms_event_id:",
    sms["sms_event_id"].duplicated().sum()
)

print("\nSMS event types:")
display(
    sms["event_type"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nSMS template codes:")
display(
    sms["template_code"]
    .value_counts(dropna=False)
    .to_frame("count")
)

Total SMS rows: 45000
Unique sms_event_id: 45000
Duplicate sms_event_id: 0

SMS event types:


,count
event_type,
SENT,11391
FAILED,11251
DELIVERED,11219
CLICKED,11139



SMS template codes:


,count
template_code,
PAYMENT_LINK,11420
DUE_REMINDER,11256
NOTICE,11166
PTP_REMINDER,11158


In [24]:
# STEP 11H: Validate SMS event IDs

print("Total SMS rows:", len(sms))

print(
    "Unique sms_event_id:",
    sms["sms_event_id"].nunique()
)

print(
    "Duplicate sms_event_id:",
    sms["sms_event_id"].duplicated().sum()
)

print("\nSMS event types:")
display(
    sms["event_type"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nSMS template codes:")
display(
    sms["template_code"]
    .value_counts(dropna=False)
    .to_frame("count")
)

Total SMS rows: 45000
Unique sms_event_id: 45000
Duplicate sms_event_id: 0

SMS event types:


,count
event_type,
SENT,11391
FAILED,11251
DELIVERED,11219
CLICKED,11139



SMS template codes:


,count
template_code,
PAYMENT_LINK,11420
DUE_REMINDER,11256
NOTICE,11166
PTP_REMINDER,11158


In [25]:
# STEP 11H: Save cleaned SMS events

clean_sms = sms.copy()

clean_sms.to_csv(
    CLEAN_DIR / "sms_events_clean.csv",
    index=False
)

print("Saved:", CLEAN_DIR / "sms_events_clean.csv")

Saved: ..\data\cleaned\sms_events_clean.csv


In [26]:
# STEP 11I: Inspect WhatsApp events

whatsapp = pd.read_csv(
    RAW_DIR / "whatsapp_events.csv"
)

print("Raw WhatsApp rows:", len(whatsapp))

print("\nColumns:")
print(whatsapp.columns.tolist())

print("\nExact duplicate rows:",
      whatsapp.duplicated().sum())

print("\nMissing values:")
print(whatsapp.isna().sum())

print("\nSample records:")
display(whatsapp.head(10));

Raw WhatsApp rows: 60600

Columns:
['whatsapp_event_id', 'account_id', 'borrower_id', 'event_at', 'message_id', 'event_type', 'template_code', 'provider_id']

Exact duplicate rows: 600

Missing values:
whatsapp_event_id    0
account_id           0
borrower_id          0
event_at             0
message_id           0
event_type           0
template_code        0
provider_id          0
dtype: int64

Sample records:


,whatsapp_event_id,account_id,borrower_id,event_at,message_id,event_type,template_code,provider_id
0,WHATSAPP_EVENT0000001,ACC0006134,BRW0007859,2026-02-27 17:49:38,MSG000008171,FAILED,FIELD_VISIT,VND0000008
1,WHATSAPP_EVENT0000002,ACC0027479,BRW0003963,2026-02-09 11:55:18,MSG000000667,FAILED,PTP_01,VND0000003
2,WHATSAPP_EVENT0000003,ACC0008356,BRW0008340,2026-07-08 05:16:13,MSG000023608,DELIVERED,PAYMENT_LINK,VND0000001
3,WHATSAPP_EVENT0000004,ACC0024727,BRW0001572,2026-05-20 19:48:52,MSG000045438,DELIVERED,PAYMENT_LINK,VND0000010
4,WHATSAPP_EVENT0000005,ACC0024918,BRW0003130,2026-01-10 10:37:24,MSG000041041,READ,LEGAL_NOTICE,VND0000015
5,WHATSAPP_EVENT0000006,ACC0005604,BRW0011815,2026-01-25 03:09:28,MSG000041619,FAILED,FIELD_VISIT,VND0000011
6,WHATSAPP_EVENT0000007,ACC0017199,BRW0006945,2026-04-11 18:37:05,MSG000035477,PAYMENT_CLICK,PTP_01,VND0000009
7,WHATSAPP_EVENT0000008,ACC0001362,BRW0011138,2026-03-14 03:43:01,MSG000011302,REPLIED,REMINDER_01,VND0000010
8,WHATSAPP_EVENT0000009,ACC0002565,BRW0001826,2026-03-04 16:41:19,MSG000011376,SENT,REMINDER_01,VND0000009
9,WHATSAPP_EVENT0000010,ACC0023647,BRW0006126,2026-01-17 16:09:00,MSG000015587,DELIVERED,LEGAL_NOTICE,VND0000007


In [27]:
# STEP 11I: Inspect WhatsApp events

whatsapp = pd.read_csv(
    RAW_DIR / "whatsapp_events.csv"
)

print("Raw WhatsApp rows:", len(whatsapp))

print("\nColumns:")
print(whatsapp.columns.tolist())

print("\nExact duplicate rows:",
      whatsapp.duplicated().sum())

print("\nMissing values:")
print(whatsapp.isna().sum())

print("\nSample records:")
display(whatsapp.head(10))

Raw WhatsApp rows: 60600

Columns:
['whatsapp_event_id', 'account_id', 'borrower_id', 'event_at', 'message_id', 'event_type', 'template_code', 'provider_id']

Exact duplicate rows: 600

Missing values:
whatsapp_event_id    0
account_id           0
borrower_id          0
event_at             0
message_id           0
event_type           0
template_code        0
provider_id          0
dtype: int64

Sample records:


,whatsapp_event_id,account_id,borrower_id,event_at,message_id,event_type,template_code,provider_id
0,WHATSAPP_EVENT0000001,ACC0006134,BRW0007859,2026-02-27 17:49:38,MSG000008171,FAILED,FIELD_VISIT,VND0000008
1,WHATSAPP_EVENT0000002,ACC0027479,BRW0003963,2026-02-09 11:55:18,MSG000000667,FAILED,PTP_01,VND0000003
2,WHATSAPP_EVENT0000003,ACC0008356,BRW0008340,2026-07-08 05:16:13,MSG000023608,DELIVERED,PAYMENT_LINK,VND0000001
3,WHATSAPP_EVENT0000004,ACC0024727,BRW0001572,2026-05-20 19:48:52,MSG000045438,DELIVERED,PAYMENT_LINK,VND0000010
4,WHATSAPP_EVENT0000005,ACC0024918,BRW0003130,2026-01-10 10:37:24,MSG000041041,READ,LEGAL_NOTICE,VND0000015
5,WHATSAPP_EVENT0000006,ACC0005604,BRW0011815,2026-01-25 03:09:28,MSG000041619,FAILED,FIELD_VISIT,VND0000011
6,WHATSAPP_EVENT0000007,ACC0017199,BRW0006945,2026-04-11 18:37:05,MSG000035477,PAYMENT_CLICK,PTP_01,VND0000009
7,WHATSAPP_EVENT0000008,ACC0001362,BRW0011138,2026-03-14 03:43:01,MSG000011302,REPLIED,REMINDER_01,VND0000010
8,WHATSAPP_EVENT0000009,ACC0002565,BRW0001826,2026-03-04 16:41:19,MSG000011376,SENT,REMINDER_01,VND0000009
9,WHATSAPP_EVENT0000010,ACC0023647,BRW0006126,2026-01-17 16:09:00,MSG000015587,DELIVERED,LEGAL_NOTICE,VND0000007


In [28]:
# STEP 11I: Validate WhatsApp event IDs

print("Total WhatsApp rows:", len(whatsapp))

print(
    "Unique whatsapp_event_id:",
    whatsapp["whatsapp_event_id"].nunique()
)

print(
    "Duplicate whatsapp_event_id:",
    whatsapp["whatsapp_event_id"].duplicated().sum()
)

print("\nWhatsApp event types:")
display(
    whatsapp["event_type"]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nWhatsApp template codes:")
display(
    whatsapp["template_code"]
    .value_counts(dropna=False)
    .to_frame("count")
)

Total WhatsApp rows: 60600
Unique whatsapp_event_id: 60000
Duplicate whatsapp_event_id: 600

WhatsApp event types:


,count
event_type,
SENT,10244
PAYMENT_CLICK,10180
FAILED,10114
REPLIED,10035
READ,10022
DELIVERED,10005



WhatsApp template codes:


,count
template_code,
PTP_01,12275
FIELD_VISIT,12177
PAYMENT_LINK,12152
LEGAL_NOTICE,12070
REMINDER_01,11926


In [29]:
# STEP 11I-A: Investigate duplicate WhatsApp event IDs

duplicate_ids = (
    whatsapp["whatsapp_event_id"]
    .value_counts()
    .loc[lambda x: x > 1]
    .index
)

duplicate_whatsapp = whatsapp[
    whatsapp["whatsapp_event_id"].isin(duplicate_ids)
].copy()

print("Duplicated WhatsApp event IDs:", len(duplicate_ids))
print("Rows involved:", len(duplicate_whatsapp))

conflict_check = (
    duplicate_whatsapp
    .groupby("whatsapp_event_id")
    .agg(
        record_count=("whatsapp_event_id", "size"),
        account_count=("account_id", "nunique"),
        borrower_count=("borrower_id", "nunique"),
        message_count=("message_id", "nunique"),
        event_time_count=("event_at", "nunique"),
        type_count=("event_type", "nunique"),
        template_count=("template_code", "nunique"),
        provider_count=("provider_id", "nunique")
    )
    .reset_index()
)

conflict_check["has_conflict"] = (
    (conflict_check["account_count"] > 1) |
    (conflict_check["borrower_count"] > 1) |
    (conflict_check["message_count"] > 1) |
    (conflict_check["event_time_count"] > 1) |
    (conflict_check["type_count"] > 1) |
    (conflict_check["template_count"] > 1) |
    (conflict_check["provider_count"] > 1)
)

print(
    "WhatsApp event IDs with conflicting fields:",
    conflict_check["has_conflict"].sum()
)

display(
    conflict_check[
        conflict_check["has_conflict"]
    ].head(20)
)

Duplicated WhatsApp event IDs: 600
Rows involved: 1200
WhatsApp event IDs with conflicting fields: 0


,whatsapp_event_id,record_count,account_count,borrower_count,message_count,event_time_count,type_count,template_count,provider_count,has_conflict


In [30]:
# STEP 11I-B: Create cleaned WhatsApp table

clean_whatsapp = whatsapp.drop_duplicates(
    subset=["whatsapp_event_id"],
    keep="first"
).copy()

print("Raw WhatsApp rows:", len(whatsapp))
print("Clean WhatsApp rows:", len(clean_whatsapp))
print("Rows removed:", len(whatsapp) - len(clean_whatsapp))
print(
    "Unique event IDs after cleaning:",
    clean_whatsapp["whatsapp_event_id"].nunique()
)

Raw WhatsApp rows: 60600
Clean WhatsApp rows: 60000
Rows removed: 600
Unique event IDs after cleaning: 60000


In [31]:
clean_whatsapp.to_csv(
    CLEAN_DIR / "whatsapp_events_clean.csv",
    index=False
)

print("Saved:", CLEAN_DIR / "whatsapp_events_clean.csv")

Saved: ..\data\cleaned\whatsapp_events_clean.csv


In [32]:
# STEP 11J: Inspect campaigns

campaigns = pd.read_csv(
    RAW_DIR / "campaigns.csv"
)

print("Raw campaign rows:", len(campaigns))

print("\nColumns:")
print(campaigns.columns.tolist())

print("\nExact duplicate rows:",
      campaigns.duplicated().sum())

print("\nMissing values:")
print(campaigns.isna().sum())

print("\nSample records:")
display(campaigns.head(15))


Raw campaign rows: 120

Columns:
['campaign_id', 'campaign_name', 'channel', 'strategy_version', 'start_at', 'target_definition', 'end_at']

Exact duplicate rows: 0

Missing values:
campaign_id          0
campaign_name        0
channel              0
strategy_version     0
start_at             0
target_definition    0
end_at               0
dtype: int64

Sample records:


,campaign_id,campaign_name,channel,strategy_version,start_at,target_definition,end_at
0,CMP0000001,DIGITAL_FOLLOWUP,FIELD,legacy,2026-02-17 06:56:01,DPD>=30,2026-04-24 06:56:01
1,CMP0000002,BOUNCE,MIXED,v2,2026-04-30 17:51:31,DPD>=60,2026-05-18 17:51:31
2,CMP0000003,30DPD_W1,MIXED,v1,2026-04-11 16:02:51,DPD>=30,2026-05-18 16:02:51
3,CMP0000004,BOUNCE,VOICE,legacy,2026-04-28 06:15:30,DPD>=60,2026-05-29 06:15:30
4,CMP0000005,60DPD_INTENT,WHATSAPP,legacy,2026-05-04 04:16:21,DPD>=60,2026-07-08 04:16:21
5,CMP0000006,BOUNCE,MIXED,legacy,2026-01-23 09:30:10,DPD>=60,2026-04-13 09:30:10
6,CMP0000007,DIGITAL_FOLLOWUP,MIXED,v2,2026-02-14 15:20:05,PROMISE_BROKEN,2026-04-22 15:20:05
7,CMP0000008,NPA_RECOVERY,SMS,v2,2026-05-21 10:58:25,PROMISE_BROKEN,2026-06-27 10:58:25
8,CMP0000009,60DPD_INTENT,WHATSAPP,v1,2026-04-18 15:47:29,HIGH_RISK,2026-06-07 15:47:29
9,CMP0000010,30DPD_W1,FIELD,v1,2026-02-14 13:09:24,DPD>=30,2026-04-30 13:09:24


In [33]:
# STEP 11J: Validate and save campaigns

print("Total campaigns:", len(campaigns))
print("Unique campaign IDs:", campaigns["campaign_id"].nunique())
print("Duplicate campaign IDs:", campaigns["campaign_id"].duplicated().sum())

print("\nStrategy versions:")
display(campaigns["strategy_version"].value_counts())

print("\nChannels:")
display(campaigns["channel"].value_counts())

print("\nTarget definitions:")
display(campaigns["target_definition"].value_counts())

Total campaigns: 120
Unique campaign IDs: 120
Duplicate campaign IDs: 0

Strategy versions:


strategy_version
legacy    37
v3        32
v2        27
v1        24
Name: count, dtype: int64


Channels:


channel
WHATSAPP    31
SMS         28
MIXED       23
VOICE       20
FIELD       18
Name: count, dtype: int64


Target definitions:


target_definition
DPD>=60           31
HIGH_RISK         25
PROMISE_BROKEN    24
DPD>=30           22
NPA               18
Name: count, dtype: int64

In [34]:
clean_campaigns = campaigns.copy()

clean_campaigns.to_csv(
    CLEAN_DIR / "campaigns_clean.csv",
    index=False
)

print("Saved:", CLEAN_DIR / "campaigns_clean.csv")

Saved: ..\data\cleaned\campaigns_clean.csv


In [36]:
clean_payments = payments.drop_duplicates(
    subset=["payment_id"],
    keep="first"
).copy()